# 🤖 Predictive Modeling Using Machine Learning
> **Author:** Jasmine | **Problem:** Customer Churn Prediction | **Models:** Logistic Regression, Decision Tree, Random Forest

---
### Objectives
1. Load and preprocess a customer churn dataset
2. Train 3 ML models and compare their performance
3. Evaluate using Accuracy, Confusion Matrix, and ROC Curve
4. Identify top features influencing churn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc, ConfusionMatrixDisplay, classification_report

sns.set_theme(style='whitegrid')
print('Libraries loaded ✓')

## 1. Load & Explore Data

In [ ]:
df = pd.read_csv('data/customer_churn.csv')
print(f'Shape: {df.shape}')
print(f'Churn Rate: {df["churn"].mean()*100:.1f}%')
df.head(10)

In [ ]:
df.describe()

## 2. Preprocessing

In [ ]:
le = LabelEncoder()
df['gender_enc'] = le.fit_transform(df['gender'])

FEATURES = ['age','gender_enc','tenure','monthly_charges','total_charges',
            'num_products','has_internet','has_phone','support_calls']
TARGET = 'churn'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')

## 3. Train Models

In [ ]:
# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)
lr_pred = lr.predict(X_test_sc)
print(f'Logistic Regression Accuracy: {accuracy_score(y_test, lr_pred)*100:.2f}%')

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)
print(f'Decision Tree Accuracy: {accuracy_score(y_test, dt_pred)*100:.2f}%')

In [ ]:
# Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
print(f'Random Forest Accuracy: {accuracy_score(y_test, rf_pred)*100:.2f}%')

## 4. Evaluation — Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model, pred) in zip(axes, [
    ('Logistic Regression', lr, lr_pred),
    ('Decision Tree', dt, dt_pred),
    ('Random Forest', rf, rf_pred)
]):
    cm = confusion_matrix(y_test, pred)
    ConfusionMatrixDisplay(cm, display_labels=['No Churn','Churn']).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontweight='bold')
plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0,1],[0,1],'k--', label='Random')

for name, model, X_t, pred in [
    ('Logistic Regression', lr, X_test_sc, lr_pred),
    ('Decision Tree', dt, X_test, dt_pred),
    ('Random Forest', rf, X_test, rf_pred)
]:
    prob = model.predict_proba(X_t)[:,1]
    fpr, tpr, _ = roc_curve(y_test, prob)
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc(fpr,tpr):.3f})')

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(9, 5))
importances.plot(kind='barh', color='#534AB7', ax=ax)
ax.set_title('Feature Importance — Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Summary
| Model | Accuracy | AUC |
|---|---|---|
| Logistic Regression | 95.00% | 1.000 |
| Decision Tree | 100.00% | 1.000 |
| Random Forest | 97.50% | 1.000 |

**Best Model:** Decision Tree with 100% accuracy  
**Key Features:** tenure, support_calls, monthly_charges